In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install scikit-learn numpy pandas
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
base_dir = '/content/drive/MyDrive/Data'
# Cấu hình data augmentation với tách validation split
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

batch_size = 32

generator_train = train_datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

generator_val = train_datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
def build_cnn(input_shape=(224, 224, 3), num_classes=2):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model

# Khởi tạo, biên dịch và huấn luyện model
model = build_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_cnn_model.h5', save_best_only=True)
]

epochs = 30
history = model.fit(
    generator_train,
    epochs=epochs,
    validation_data=generator_val,
    callbacks=callbacks
)

# Đánh giá trên tập validation
eval_loss, eval_acc = model.evaluate(generator_val)
print(f"Validation Accuracy: {eval_acc*100:.2f}%")


Found 938 images belonging to 2 classes.
Found 234 images belonging to 2 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    44,302,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │         1,026 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,397,122 (169.36 MB)

...

30/30 ━━━━━━━━━━━━━━━━━━━━ 28s 935ms/step - accuracy: 0.7646 - loss: 0.4952 - val_accuracy: 0.7863 - val_loss: 0.4823
Epoch 24/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 518ms/step - accuracy: 0.7440 - loss: 0.5064

30/30 ━━━━━━━━━━━━━━━━━━━━ 29s 984ms/step - accuracy: 0.7439 - loss: 0.5064 - val_accuracy: 0.7650 - val_loss: 0.4798
Epoch 25/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 22s 752ms/step - accuracy: 0.7688 - loss: 0.5055 - val_accuracy: 0.7607 - val_loss: 0.5059
Epoch 26/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 22s 742ms/step - accuracy: 0.7530 - loss: 0.5223 - val_accuracy: 0.7179 - val_loss: 0.5625
Epoch 27/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 500ms/step - accuracy: 0.7870 - loss: 0.4704

30/30 ━━━━━━━━━━━━━━━━━━━━ 27s 903ms/step - accuracy: 0.7867 - loss: 0.4707 - val_accuracy: 0.7650 - val_loss: 0.4715
Epoch 28/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 500ms/step - accuracy: 0.7612 - loss: 0.4590

30/30 ━━━━━━━━━━━━━━━━━━━━ 24s 813ms/step - accuracy: 0.7612 - loss: 0.4597 - val_accuracy: 0.7692 - val_loss: 0.4710
Epoch 29/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 22s 704ms/step - accuracy: 0.7328 - loss: 0.5031 - val_accuracy: 0.7778 - val_loss: 0.4796
Epoch 30/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 503ms/step - accuracy: 0.7855 - loss: 0.4663

30/30 ━━━━━━━━━━━━━━━━━━━━ 28s 959ms/step - accuracy: 0.7854 - loss: 0.4664 - val_accuracy: 0.7821 - val_loss: 0.4386
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 761ms/step - accuracy: 0.7702 - loss: 0.4597
Validation Accuracy: 78.21%
